In [3]:
import polars as pl

# batch 8

In [4]:
import extern
extern.run("grep SRA submit_batch3 |awk '{print $2}' |cat - runlists/batch1_500.txt runlists/batch2_500.txt runlists/batch4_2000.txt runlists/batch5_test10.txt runlists/batch6_test50000.txt runlists/batch7_test10000.txt >/tmp/runs")

''

In [5]:
prev = pl.read_csv("/tmp/runs", has_header=False)
prev.columns = ["acc"]
prev.shape

(63406, 1)

In [6]:
total = pl.read_csv('runlists/acc_less_than_20k_mbytes.csv', has_header=False)
total.columns = ["acc"]
total.shape

(710928, 1)

In [7]:
batch8_possible = total.join(prev, on="acc", how="anti")
batch8_possible.shape

(647522, 1)

In [8]:
# batch8 = batch8_possible.sample(100000)
# print(batch8.shape, batch8[:5])
# batch8.write_csv('runlists/batch8_100000.txt', include_header=False)

In [9]:
# 
extern.run("grep SRA submit_batch3 |awk '{print $2}' |cat - runlists/batch1_500.txt runlists/batch2_500.txt runlists/batch4_2000.txt runlists/batch5_test10.txt runlists/batch6_test50000.txt runlists/batch7_test10000.txt runlists/batch8_100000.txt >/tmp/runs")

prev = pl.read_csv("/tmp/runs", has_header=False)
prev.columns = ["acc"]
print(prev.shape)

batch9_possible = total.join(prev, on="acc", how="anti")
batch9_possible.shape

(163406, 1)


(547522, 1)

In [10]:
# batch9_possible.write_csv('runlists/batch9_da_rest.txt', include_header=False)

# batch10 - cleaning up and everythin 10Gbp or less

In [11]:
extern.run('cat  s3_us-east2.txt s3_woodcrob-sandpiper-us-east-1.txt |grep RR |sed "s/  */\t/g" >/tmp/runs')

prev1 = pl.read_csv('/tmp/runs', has_header=False, separator='\t')
prev1.columns = ["acc", "time", "size", "path"]
prev2 = prev1.with_columns(pl.col("path").alias("acc").str.split('.').list.get(0)).select(["acc", "size"])
prev2.shape, prev2[:3]

((137852, 2),
 shape: (3, 2)
 ┌───────────┬───────┐
 │ acc       ┆ size  │
 │ ---       ┆ ---   │
 │ str       ┆ i64   │
 ╞═══════════╪═══════╡
 │ DRR001961 ┆ 26759 │
 │ DRR003630 ┆ 60975 │
 │ DRR003636 ┆ 45914 │
 └───────────┴───────┘)

In [12]:
prev2.filter(pl.col("size") == 0).shape

(4305, 2)

In [13]:
prev3 = prev2.filter(pl.col("size") > 0)
prev3.shape

(133547, 2)

In [14]:
df = pl.read_csv('~/git/sandpiper/sra_metadata/sra_metadata_20250220.some_columns.csv.gz', has_header=False)
df.columns = ['acc','releasedate','mbases','mbytes']
df[:4]

acc,releasedate,mbases,mbytes
str,str,i64,i64
"""SRR15442735""","""2021-08-13T00:00:00+00:00""",6638,2614
"""ERR1959224""","""2017-07-08T00:00:00+00:00""",8555,3195
"""ERR5003368""","""2020-12-23T00:00:00+00:00""",1013,344
"""ERR5261058""","""2021-03-15T00:00:00+00:00""",20619,6792


In [15]:
batch10_possible = df.filter(pl.col('mbases') < 8000).join(prev, on="acc", how="anti")
batch10_possible.shape, batch10_possible.sample(3)

((471178, 4),
 shape: (3, 4)
 ┌─────────────┬───────────────────────────┬────────┬────────┐
 │ acc         ┆ releasedate               ┆ mbases ┆ mbytes │
 │ ---         ┆ ---                       ┆ ---    ┆ ---    │
 │ str         ┆ str                       ┆ i64    ┆ i64    │
 ╞═════════════╪═══════════════════════════╪════════╪════════╡
 │ ERR4571283  ┆ 2020-09-15T00:00:00+00:00 ┆ 922    ┆ 363    │
 │ SRR11999378 ┆ 2021-07-02T00:00:00+00:00 ┆ 6706   ┆ 2732   │
 │ SRR5240588  ┆ 2017-02-14T00:00:00+00:00 ┆ 2064   ┆ 841    │
 └─────────────┴───────────────────────────┴────────┴────────┘)

In [16]:
batch10_possible.select('acc').write_csv('runlists/batch10_10gbp_or_smaller.txt', include_header=False)

# batch11 small fast test of multiple

In [17]:
per_acc = pl.read_csv('~/m/msingle/mess/174_R220_renew/processing_20240531/per_acc_summary.csv')
per_acc.shape, per_acc[:3]

((245436, 11),
 shape: (3, 11)
 ┌────────────┬───────────┬───────────┬───────────┬───┬─────────┬───────────┬───────────┬───────────┐
 │ sample     ┆ root_cove ┆ species_c ┆ bacterial ┆ … ┆ warning ┆ predictio ┆ organism  ┆ host_or_n │
 │ ---        ┆ rage      ┆ overage   ┆ _archaeal ┆   ┆ ---     ┆ n         ┆ ---       ┆ ot        │
 │ str        ┆ ---       ┆ ---       ┆ _bases    ┆   ┆ str     ┆ ---       ┆ str       ┆ ---       │
 │            ┆ f64       ┆ f64       ┆ ---       ┆   ┆         ┆ str       ┆           ┆ str       │
 │            ┆           ┆           ┆ i64       ┆   ┆         ┆           ┆           ┆           │
 ╞════════════╪═══════════╪═══════════╪═══════════╪═══╪═════════╪═══════════╪═══════════╪═══════════╡
 │ SRR8634435 ┆ 345.4     ┆ 0.933237  ┆ 117232064 ┆ … ┆ null    ┆ host      ┆ feces met ┆ host      │
 │            ┆           ┆           ┆ 2         ┆   ┆         ┆           ┆ agenome   ┆           │
 │ SRR8640623 ┆ 730.5     ┆ 0.127748  ┆ 139594647 ┆

In [18]:
batch11 = batch10_possible.join(per_acc.filter(pl.col('root_coverage')>0), left_on='acc', right_on='sample', how='inner').sort('mbases').head(4)

In [19]:
# batch11.select('acc').write_csv('runlists/batch11_4.txt', include_header=False)

# batch 12 - 8 to 30Gbp

In [20]:
df = pl.read_csv('~/git/sandpiper/sra_metadata/sra_metadata_20250220.some_columns.csv.gz', has_header=False)
df.columns = ['acc','releasedate','mbases','mbytes']
df[:4]

acc,releasedate,mbases,mbytes
str,str,i64,i64
"""SRR15442735""","""2021-08-13T00:00:00+00:00""",6638,2614
"""ERR1959224""","""2017-07-08T00:00:00+00:00""",8555,3195
"""ERR5003368""","""2020-12-23T00:00:00+00:00""",1013,344
"""ERR5261058""","""2021-03-15T00:00:00+00:00""",20619,6792


In [21]:
# how much 0-8, 8-15, 15-30 and 30+ mbases?
(
    df.select('mbases').sum()[0,0] /1e9,
    df.filter(pl.col('mbases') < 8000).select('mbases').sum()[0,0] /1e9,
    df.filter(pl.col('mbases') < 15000).filter(pl.col('mbases') >= 8000).select('mbases').sum()[0,0] /1e9,
    df.filter(pl.col('mbases') < 30000).filter(pl.col('mbases') >= 15000).select('mbases').sum()[0,0] /1e9,
    df.filter(pl.col('mbases') >= 30000).select('mbases').sum()[0,0] /1e9,
)


(4.866907013, 1.763913074, 1.36664454, 0.898161387, 0.838188012)

In [22]:
# Try those < 8gbases again where they failed - maybe the increased RAM of a this batch will get them through?
extern.run('cat s3_ls/* |grep RR |sed "s/  */\t/g" >/tmp/runs')

prev4 = pl.read_csv('/tmp/runs', has_header=False, separator='\t')
prev4.columns = ["acc", "time", "size", "path"]
print(prev4.group_by(pl.col('size') > 0).len())
prev5 = prev4.filter(pl.col('size') > 0).with_columns(pl.col("path").alias("acc").str.split('.').list.get(0))
prev5.shape

shape: (2, 2)
┌───────┬────────┐
│ size  ┆ len    │
│ ---   ┆ ---    │
│ bool  ┆ u32    │
╞═══════╪════════╡
│ false ┆ 8769   │
│ true  ┆ 591446 │
└───────┴────────┘


(591446, 4)

In [23]:
# Create a batch of 1000 runs with 8-30 Gbps, that aren't in any previous 
batch12_possible = df.filter(pl.col('mbases') < 30000).filter(pl.col('mbases') >= 8000).join(prev5, on="acc", how="anti")
batch12_possible.shape[0], batch12_possible.sample(5), batch12_possible.select(pl.col('mbases')).sum()[0,0] /1e9

(146031,
 shape: (5, 4)
 ┌─────────────┬───────────────────────────┬────────┬────────┐
 │ acc         ┆ releasedate               ┆ mbases ┆ mbytes │
 │ ---         ┆ ---                       ┆ ---    ┆ ---    │
 │ str         ┆ str                       ┆ i64    ┆ i64    │
 ╞═════════════╪═══════════════════════════╪════════╪════════╡
 │ SRR3624175  ┆ 2017-08-31T00:00:00+00:00 ┆ 9822   ┆ 4674   │
 │ SRR13774545 ┆ 2021-02-24T00:00:00+00:00 ┆ 9591   ┆ 2806   │
 │ ERR11596044 ┆ 2025-01-02T00:00:00+00:00 ┆ 8960   ┆ 3081   │
 │ ERR4398820  ┆ 2020-09-08T00:00:00+00:00 ┆ 22875  ┆ 6405   │
 │ SRR32192472 ┆ 2025-02-04T00:00:00+00:00 ┆ 10777  ┆ 3606   │
 └─────────────┴───────────────────────────┴────────┴────────┘,
 1.957546801)

In [24]:
# batch12_possible.sample(fraction=1, shuffle=True).select('acc').write_csv('runlists/batch12_8_to_30gbp.txt', include_header=False)

# batch 13 - 30gbp and above

In [26]:
batch13_possible = df.filter(pl.col('mbases') >= 30000).join(prev5, on="acc", how="anti")
batch13_possible.shape[0], batch13_possible.sample(5), batch13_possible.select(pl.col('mbases')).sum()[0,0] /1e9

(16011,
 shape: (5, 4)
 ┌─────────────┬───────────────────────────┬────────┬────────┐
 │ acc         ┆ releasedate               ┆ mbases ┆ mbytes │
 │ ---         ┆ ---                       ┆ ---    ┆ ---    │
 │ str         ┆ str                       ┆ i64    ┆ i64    │
 ╞═════════════╪═══════════════════════════╪════════╪════════╡
 │ ERR11547419 ┆ 2023-09-01T00:00:00+00:00 ┆ 31286  ┆ 9933   │
 │ SRR14887735 ┆ 2021-06-23T00:00:00+00:00 ┆ 38030  ┆ 10718  │
 │ ERR3589572  ┆ 2019-10-22T00:00:00+00:00 ┆ 40878  ┆ 27139  │
 │ SRR15910078 ┆ 2022-09-26T00:00:00+00:00 ┆ 114602 ┆ 39812  │
 │ SRR32129091 ┆ 2025-01-26T00:00:00+00:00 ┆ 51189  ┆ 16915  │
 └─────────────┴───────────────────────────┴────────┴────────┘,
 0.798983057)

In [ ]:
# batch13_possible.sample(fraction=1, shuffle=True).select('acc').write_csv('runlists/batch13_30gbp_plus.txt', include_header=False)